In [ ]:
#1 Installed libraries that allow: importing a translation model, a tokenizer and an evaluation tool (BLEU).
!pip install transformers sentencepiece sacrebleu

In [ ]:
# Uploaded the csv file, which contains extracted ENG-SLO pairs from the tmx files.
from google.colab import files
uploaded = files.upload()

In [ ]:
# Converts the csv into a dataframe.
import pandas as pd

df = pd.read_csv("dgt_all.csv")

print(df.head())
print(len(df))

In [ ]:
# Extracts a sample of 500 (couldn't do 1000, cauz a later cell takes more than an hour to run and then crashes either way ); ) for the training set, and a sample of 200 for the test set (which will not be used now, but later).
train_df = df.sample(500, random_state=42)
test_df = df.drop(train_df.index).sample(200, random_state=42)

print(len(train_df), len(test_df))

500 200


In [ ]:
# Imports tools for tokenization and the NLLB translation model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

fb_model = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(fb_model)
model = AutoModelForSeq2SeqLM.from_pretrained(fb_model)

In [ ]:
# Made the model run on GPU instead of CPU, otherwise it didn't want to run at all even for 500 samples.
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

In [ ]:
# Defines the source and target languages for the model to translate.
tokenizer.src_lang = "eng_Latn"
target_lang = "slv_Latn"

In [ ]:
# Makes the model translate more than just 1 sentence at a time (for speed increase).
def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_lang)
    )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [ ]:
# Makes the model translate 8 sentences at once. The model translates the ENG sentences into SLO and appends that to the list PREDICTIONS, and to REFERENCES we append the
# already translated SLO sentences, this is our Gold Standard.
batch_size = 8

predictions = []
references = []

for i in range(0, len(train_df), batch_size):
    batch = train_df.iloc[i:i+batch_size]

    en_texts = batch["en"].tolist()
    sl_texts = batch["sl"].tolist()

    preds = translate_batch(en_texts)

    predictions.extend(preds)
    references.extend(sl_texts)

In [ ]:
# Just checking if everything works by printing 20 examples: the Model translation and how it compares to the Gold Standard.
for pred, ref in list(zip(predictions, references))[:20]:
  print('Model translation:', pred)
  print('Gold standard:', ref)
  print('-' * 40)

In [ ]:
# Evaluating the model's translation compared to the Gold standard.
import sacrebleu

bleu = sacrebleu.corpus_bleu(predictions, [references])

print(f'BLEU score:, {bleu.score:.2f}')

BLEU score:, 36.17
